# Auto Insurance Pure Premium Model
**Frequency × severity GLMs, a Bayesian regional adjustment, and model validation on French motor insurance data**

**Question:** how much should a driver's policy cost, before expenses and profit, given who they are and what they drive?

**Pure premium = expected claim frequency (claims per year) × expected claim severity (€ per claim).**

**Data:** freMTPL2freq and freMTPL2sev (French motor third-party liability, about 678,000 policies), loaded from OpenML.

**Plan:**
1. Setup
2. Load and merge the data
3. Clean the data
4. Explore claim patterns
5. Bin rating factors
6. Train/test split
7. Frequency GLM (Poisson)
8. Severity GLM (Gamma)
9. Bayesian regional adjustment (PyMC)
10. Benchmark models (Tweedie GLM, gradient boosting)
11. Validation: Gini, Lorenz curve, lift chart
12. Relativity table and top risk drivers
13. Save the model files for the Streamlit app
14. Results summary for your README and resume

**How to run:** Runtime → Run all. The whole notebook takes about 5–10 minutes on the free Colab CPU (the PyMC step is the slowest). No GPU is needed.

## 1. Setup
Imports, a fixed random seed for reproducibility, and an output folder. PyMC and ArviZ come preinstalled on Colab; the cell installs them only if they're missing.

In [ ]:
import importlib
import json
import os
import subprocess
import sys
import time
import warnings

warnings.filterwarnings("ignore")


def ensure(pip_name, import_name=None):
    """Import a package, installing it with pip first if needed. Returns True if it's available."""
    try:
        importlib.import_module(import_name or pip_name)
        return True
    except ImportError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
            importlib.import_module(import_name or pip_name)
            return True
        except Exception as err:  # no internet, build failure, etc.
            print(f"Could not install {pip_name}: {err}")
            return False


HAS_PYMC = ensure("pymc") and ensure("arviz")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PoissonRegressor, GammaRegressor, TweedieRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_poisson_deviance, auc
from scipy import stats

SEED = 42
np.random.seed(SEED)
OUT = "artifacts"
os.makedirs(OUT, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("PyMC available:", HAS_PYMC)

## 2. Load and merge the data
- **freMTPL2freq** has one row per policy: exposure (fraction of a year insured), number of claims, and rating factors.
- **freMTPL2sev** has one row per claim with its cost. We sum claim costs per policy and join them onto the policy table.

In [ ]:
t0 = time.time()
freq_raw = fetch_openml(data_id=41214, as_frame=True, parser="auto").frame
sev_raw = fetch_openml(data_id=41215, as_frame=True, parser="auto").frame
print(f"Downloaded in {time.time() - t0:.0f}s")
print("Policies table:", freq_raw.shape, "| Claims table:", sev_raw.shape)

freq_raw["IDpol"] = pd.to_numeric(freq_raw["IDpol"]).astype(int)
sev_raw["IDpol"] = pd.to_numeric(sev_raw["IDpol"]).astype(int)
claim_cost = sev_raw.groupby("IDpol")["ClaimAmount"].sum()

df = freq_raw.set_index("IDpol").join(claim_cost, how="left")
df["ClaimAmount"] = df["ClaimAmount"].fillna(0.0)

# Text columns sometimes come wrapped in quotes ('B12'), so strip them.
for col in ["Area", "VehBrand", "VehGas", "Region"]:
    df[col] = df[col].astype(str).str.strip("'")
for col in ["ClaimNb", "Exposure", "VehPower", "VehAge", "DrivAge", "BonusMalus", "Density", "ClaimAmount"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.head()

## 3. Clean the data
Each rule is documented so it can be explained in an interview:
1. **Claims with no recorded cost** (ClaimNb ≥ 1 but ClaimAmount = 0): the claims table has no cost for these, so we set ClaimNb to 0. That keeps frequency and severity consistent.
2. **Costs with no recorded claim** (ClaimAmount > 0 but ClaimNb = 0): inconsistent, so we drop these few rows.
3. **Exposure capped at 1 year**: a few policies show more than one year, which is a data error.
4. **ClaimNb capped at 4**: a handful of policies show up to 16 claims, likely data errors that would distort the frequency model.
5. **Claim cost per policy capped at €200,000**: very rare, huge claims dominate a severity model. In practice, large losses are priced separately with a large-loss load.

In [ ]:
n_start = len(df)
log = {}
log["missing_values"] = int(df.isna().sum().sum())
df = df.dropna()

mask_no_cost = (df["ClaimNb"] >= 1) & (df["ClaimAmount"] == 0)
log["claims_without_cost_set_to_0"] = int(mask_no_cost.sum())
df.loc[mask_no_cost, "ClaimNb"] = 0

mask_no_claim = (df["ClaimAmount"] > 0) & (df["ClaimNb"] == 0)
log["cost_without_claim_dropped"] = int(mask_no_claim.sum())
df = df.loc[~mask_no_claim].copy()

log["exposure_capped"] = int((df["Exposure"] > 1).sum())
log["claimnb_capped"] = int((df["ClaimNb"] > 4).sum())
log["claimamount_capped"] = int((df["ClaimAmount"] > 200_000).sum())
df["Exposure"] = df["Exposure"].clip(upper=1.0)
df["ClaimNb"] = df["ClaimNb"].clip(upper=4)
df["ClaimAmount"] = df["ClaimAmount"].clip(upper=200_000)
df = df.loc[df["Exposure"] > 0].copy()

df["Frequency"] = df["ClaimNb"] / df["Exposure"]        # claims per policy-year
df["PurePremium"] = df["ClaimAmount"] / df["Exposure"]  # € per policy-year

print("Cleaning log:")
for k, v in log.items():
    print(f"  {k:32s} {v:,}")
print(f"\nPolicies: {n_start:,} → {len(df):,}")

portfolio = {
    "n_policies": int(len(df)),
    "total_exposure_years": float(df["Exposure"].sum()),
    "n_claims": int(df["ClaimNb"].sum()),
    "claim_frequency": float(df["ClaimNb"].sum() / df["Exposure"].sum()),
    "avg_severity": float(df["ClaimAmount"].sum() / df["ClaimNb"].sum()),
    "pure_premium": float(df["ClaimAmount"].sum() / df["Exposure"].sum()),
}
print(f"Total exposure:      {portfolio['total_exposure_years']:,.0f} policy-years")
print(f"Claim frequency:     {portfolio['claim_frequency']:.4f} claims per year")
print(f"Average severity:    €{portfolio['avg_severity']:,.0f} per claim")
print(f"Pure premium:        €{portfolio['pure_premium']:,.2f} per policy-year")

## 4. Explore claim patterns
Claim frequency must be computed as **total claims ÷ total exposure**, not as an average of per-policy frequencies, because short policies would otherwise get too much weight.

In [ ]:
def one_way(data, col, bins=None):
    """Exposure-weighted claim frequency, severity, and pure premium by one variable."""
    key = pd.cut(data[col], bins) if bins is not None else data[col]
    g = data.groupby(key, observed=True).agg(
        Exposure=("Exposure", "sum"), Claims=("ClaimNb", "sum"), Cost=("ClaimAmount", "sum"))
    g["Frequency"] = g["Claims"] / g["Exposure"]
    g["Severity"] = g["Cost"] / g["Claims"].replace(0, np.nan)
    g["PurePremium"] = g["Cost"] / g["Exposure"]
    return g


fig, axes = plt.subplots(2, 3, figsize=(15, 8))
specs = [
    ("DrivAge", [17, 21, 25, 30, 40, 50, 60, 70, 100], "Driver age"),
    ("BonusMalus", [49, 50, 60, 75, 100, 250], "Bonus-malus"),
    ("VehAge", [-1, 0, 2, 5, 10, 15, 100], "Vehicle age"),
    ("VehPower", [3, 5, 6, 7, 9, 11, 15], "Vehicle power"),
    ("Density", [0, 50, 200, 500, 2000, 10000, 30000], "Population density"),
    ("VehGas", None, "Fuel type"),
]
for ax, (col, bins, title) in zip(axes.flat, specs):
    g = one_way(df, col, bins)
    ax.bar(g.index.astype(str), g["Frequency"], color="#2B6F77")
    ax.axhline(portfolio["claim_frequency"], color="#C0392B", ls="--", lw=1, label="Portfolio")
    ax.set_title(f"Claim frequency by {title.lower()}")
    ax.tick_params(axis="x", rotation=35)
axes[0, 0].legend()
plt.tight_layout()
plt.show()

claims_only = df.loc[df["ClaimNb"] > 0]
sev_per_claim = claims_only["ClaimAmount"] / claims_only["ClaimNb"]
plt.figure(figsize=(8, 3.5))
plt.hist(np.log10(sev_per_claim), bins=80, color="#2B6F77")
plt.xlabel("log10(cost per claim, €)")
plt.ylabel("Number of policies")
plt.title("Claim severity is heavily right-skewed (why we use a Gamma model)")
plt.show()
print(f"Median cost per claim: €{sev_per_claim.median():,.0f} | Mean: €{sev_per_claim.mean():,.0f}")

## 5. Bin rating factors
Insurers usually price with **bands** (rating tables) rather than raw numbers, because:
- many effects aren't linear (young and very old drivers are both riskier than middle-aged drivers),
- bands give a readable relativity table that a pricing team or regulator can review.

Each rule is `(label, upper bound)`. A value goes in the first band whose upper bound it doesn't exceed, and `None` means "no upper limit." The same rules are saved for the Streamlit app, so the app bins inputs exactly the way the model was trained.

**Area** is left out because it's almost a recoded version of population density, and including both would split one effect across two correlated variables. **Region** is handled separately by the Bayesian model in Section 9.

In [ ]:
BAND_RULES = {
    "DrivAgeBand": ("DrivAge", [("18-21", 21), ("22-25", 25), ("26-30", 30), ("31-40", 40),
                                ("41-50", 50), ("51-60", 60), ("61-70", 70), ("71+", None)]),
    "VehAgeBand": ("VehAge", [("0", 0), ("1-2", 2), ("3-5", 5), ("6-10", 10), ("11-15", 15), ("16+", None)]),
    "VehPowerBand": ("VehPower", [("4-5", 5), ("6", 6), ("7", 7), ("8-9", 9), ("10-11", 11), ("12+", None)]),
    "BonusMalusBand": ("BonusMalus", [("50", 50), ("51-60", 60), ("61-75", 75), ("76-100", 100), ("101+", None)]),
    "DensityBand": ("Density", [("1-50", 50), ("51-200", 200), ("201-500", 500), ("501-2000", 2000),
                                ("2001-10000", 10000), ("10000+", None)]),
}
CATEGORICAL = ["VehBrand", "VehGas"]
FACTORS = list(BAND_RULES) + CATEGORICAL  # rating factors used in the GLMs

REGION_NAMES = {
    "R11": "Île-de-France", "R21": "Champagne-Ardenne", "R22": "Picardie", "R23": "Haute-Normandie",
    "R24": "Centre", "R25": "Basse-Normandie", "R26": "Bourgogne", "R31": "Nord-Pas-de-Calais",
    "R41": "Lorraine", "R42": "Alsace", "R43": "Franche-Comté", "R52": "Pays de la Loire",
    "R53": "Bretagne", "R54": "Poitou-Charentes", "R72": "Aquitaine", "R73": "Midi-Pyrénées",
    "R74": "Limousin", "R82": "Rhône-Alpes", "R83": "Auvergne", "R91": "Languedoc-Roussillon",
    "R93": "Provence-Alpes-Côte d'Azur", "R94": "Corse",
}


def apply_bands(values, rules):
    """Assign each value to its band label using (label, upper bound) rules."""
    labels = [r[0] for r in rules]
    edges = [-np.inf] + [r[1] for r in rules[:-1]] + [np.inf]
    return pd.cut(values, bins=edges, labels=labels, right=True).astype(str)


for band_col, (raw_col, rules) in BAND_RULES.items():
    df[band_col] = apply_bands(df[raw_col], rules)


def natural_key(s):
    """Sort B1, B2, ..., B10 in number order instead of text order."""
    digits = "".join(ch for ch in s if ch.isdigit())
    return (int(digits) if digits else 0, s)


LEVELS = {f: [r[0] for r in BAND_RULES[f][1]] for f in BAND_RULES}
for f in CATEGORICAL:
    LEVELS[f] = sorted(df[f].unique(), key=natural_key)
REGIONS = sorted(df["Region"].unique(), key=natural_key)

for f in FACTORS:
    counts = df.groupby(f)["Exposure"].sum().reindex(LEVELS[f])
    print(f"{f:15s}", {k: f"{v / counts.sum():.1%}" for k, v in counts.items()})

## 6. Train/test split
80% of policies train the models, and 20% are held out to judge them honestly. The **baseline level** of each factor is its most common level (by exposure in training), so the baseline driver is a typical driver and every relativity reads as "compared with a typical driver."

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=SEED)
train, test = train.copy(), test.copy()
print(f"Train: {len(train):,} policies | Test: {len(test):,} policies")

BASE = {f: train.groupby(f)["Exposure"].sum().idxmax() for f in FACTORS}
print("Baseline driver:", BASE)


def design_matrix(data, factors=FACTORS, levels=LEVELS, base=BASE):
    """One-hot (dummy) columns for every non-baseline level. Column names look like 'Factor=level'."""
    cols, names = [], []
    for f in factors:
        vals = data[f].astype(str).to_numpy()
        for lv in levels[f]:
            if lv == base[f]:
                continue
            cols.append((vals == lv).astype(np.float64))
            names.append(f"{f}={lv}")
    return pd.DataFrame(np.column_stack(cols), columns=names, index=data.index)


X_train = design_matrix(train)
X_test = design_matrix(test)
print("Design matrix:", X_train.shape)

## 7. Frequency GLM (Poisson, log link)
- **Why Poisson:** claim counts are small non-negative whole numbers.
- **Why exposure weights:** a policy insured for 3 months has a quarter of a year to produce claims. Fitting claims per year (`ClaimNb / Exposure`) with `Exposure` as the weight is mathematically the same as a Poisson GLM with a `log(Exposure)` offset.
- **Why a log link:** effects multiply, so each coefficient becomes a relativity: `exp(β) = 1.25` means "25% more claims than the baseline level."

In [ ]:
t0 = time.time()
freq_glm = PoissonRegressor(alpha=0.0, solver="newton-cholesky", max_iter=300)
freq_glm.fit(X_train, train["Frequency"], sample_weight=train["Exposure"])
print(f"Fitted in {time.time() - t0:.1f}s")

FREQ_COEF = dict(zip(X_train.columns, freq_glm.coef_))
FREQ_INTERCEPT = float(freq_glm.intercept_)

train["freq_glm"] = freq_glm.predict(X_train)
test["freq_glm"] = freq_glm.predict(X_test)

# Balance check: a Poisson GLM with an intercept reproduces the total training claims.
print(f"Predicted training claims: {(train['freq_glm'] * train['Exposure']).sum():,.0f} "
      f"| Actual: {train['ClaimNb'].sum():,.0f}")
print(f"Test Poisson deviance (GLM):   "
      f"{mean_poisson_deviance(test['Frequency'], test['freq_glm'], sample_weight=test['Exposure']):.4f}")
null_pred = np.full(len(test), train["ClaimNb"].sum() / train["Exposure"].sum())
print(f"Test Poisson deviance (no model): "
      f"{mean_poisson_deviance(test['Frequency'], null_pred, sample_weight=test['Exposure']):.4f}")

## 8. Severity GLM (Gamma, log link)
- Fitted only on policies **with** claims: target is cost per claim, weighted by number of claims.
- **Why Gamma:** costs are positive and right-skewed, and their spread grows with their size, which is exactly what a Gamma distribution assumes.
- A very light ridge penalty (`alpha=1e-4`) keeps levels with few claims from getting extreme coefficients.

In [ ]:
tr_c = train.loc[train["ClaimNb"] > 0].copy()
tr_c["SevPerClaim"] = tr_c["ClaimAmount"] / tr_c["ClaimNb"]
X_tr_c = design_matrix(tr_c)

t0 = time.time()
sev_glm = GammaRegressor(alpha=1e-4, solver="newton-cholesky", max_iter=300)
sev_glm.fit(X_tr_c, tr_c["SevPerClaim"], sample_weight=tr_c["ClaimNb"])
print(f"Fitted on {len(tr_c):,} policies with claims in {time.time() - t0:.1f}s")

SEV_COEF = dict(zip(X_tr_c.columns, sev_glm.coef_))
SEV_INTERCEPT = float(sev_glm.intercept_)

train["sev_glm"] = sev_glm.predict(X_train)
test["sev_glm"] = sev_glm.predict(X_test)

te_c = test.loc[test["ClaimNb"] > 0]
actual_sev = te_c["ClaimAmount"].sum() / te_c["ClaimNb"].sum()
pred_sev = np.average(te_c["sev_glm"], weights=te_c["ClaimNb"])
print(f"Test average severity: actual €{actual_sev:,.0f} | predicted €{pred_sev:,.0f}")

## 9. Bayesian regional adjustment (PyMC)
Some of France's 22 regions have lots of data and some have very little. A raw "actual ÷ expected" ratio for a small region is mostly noise. A **hierarchical Bayesian model** shrinks small regions toward the national average and lets large regions speak for themselves (partial pooling).

**Model**, using region-level totals so it runs in seconds:
- Expected claims for region *r*, `E_r`, come from the frequency GLM (which doesn't use region), so the regional factor only captures what the other factors don't explain.
- `Claims_r ~ Poisson(E_r × m_r)`, with `m_r = exp(u_r)`, `u_r ~ Normal(0, σ)`, `σ ~ HalfNormal(0.3)`.
- `m_r` is the regional multiplier: 1.10 means 10% more claims than expected.

If PyMC isn't available, the cell falls back to the conjugate **Poisson–Gamma** (Bühlmann-style credibility) model, which gives the same kind of shrinkage with a closed-form answer.

In [ ]:
reg = (train.assign(Expected=train["freq_glm"] * train["Exposure"])
       .groupby("Region").agg(Exposure=("Exposure", "sum"), Claims=("ClaimNb", "sum"),
                              Expected=("Expected", "sum"))
       .reindex(REGIONS))
reg["RawRatio"] = reg["Claims"] / reg["Expected"]
observed = reg["Claims"].to_numpy().astype(int)
expected = reg["Expected"].to_numpy()

REGION_METHOD = None
if HAS_PYMC:
    try:
        import pymc as pm
        import arviz as az

        t0 = time.time()
        with pm.Model() as region_model:
            sigma = pm.HalfNormal("sigma", sigma=0.3)
            z = pm.Normal("z", 0.0, 1.0, shape=len(REGIONS))  # non-centered form samples more cleanly
            mult = pm.Deterministic("mult", pm.math.exp(sigma * z))
            pm.Poisson("claims", mu=expected * mult, observed=observed)
            # Faster option if needed: draws=500, tune=500.
            idata = pm.sample(draws=1000, tune=1000, chains=2, cores=1, target_accept=0.95,
                              random_seed=SEED, progressbar=False)
        post = idata.posterior["mult"].stack(s=("chain", "draw")).to_numpy()  # (regions, samples)
        reg["PostMean"] = post.mean(axis=1)
        reg["CI_low"] = np.percentile(post, 2.5, axis=1)
        reg["CI_high"] = np.percentile(post, 97.5, axis=1)
        sigma_mean = float(idata.posterior["sigma"].mean())
        print(f"PyMC sampled in {time.time() - t0:.0f}s | σ (between-region spread) ≈ {sigma_mean:.3f}")
        print(az.summary(idata, var_names=["sigma"], round_to=3))
        REGION_METHOD = "PyMC hierarchical Poisson model (non-centered), 2 chains × 1,000 draws"
    except Exception as err:
        print("PyMC failed, using the closed-form fallback:", err)

if REGION_METHOD is None:
    # Conjugate fallback: m_r ~ Gamma(k, k) prior with mean 1; posterior Gamma(k + claims, k + expected).
    tau2 = max(np.var(reg["RawRatio"]) - np.mean(1 / expected), 1e-4)  # between-region variance
    k = 1 / tau2
    a, b = k + observed, k + expected
    reg["PostMean"] = a / b
    reg["CI_low"] = stats.gamma.ppf(0.025, a, scale=1 / b)
    reg["CI_high"] = stats.gamma.ppf(0.975, a, scale=1 / b)
    REGION_METHOD = f"Poisson–Gamma credibility model (closed form, prior strength k = {k:.0f})"

reg["RegionName"] = [REGION_NAMES.get(r, r) for r in reg.index]
reg["ExposureShare"] = reg["Exposure"] / reg["Exposure"].sum()
print("Method:", REGION_METHOD)
display_cols = ["RegionName", "Exposure", "Claims", "Expected", "RawRatio", "PostMean", "CI_low", "CI_high"]
print(reg[display_cols].sort_values("PostMean").round(3).to_string())

# Partial pooling plot: raw ratios (dots) vs Bayesian estimates (bars with 95% intervals).
r = reg.sort_values("PostMean")
fig, ax = plt.subplots(figsize=(9, 7))
ax.errorbar(r["PostMean"], range(len(r)), xerr=[r["PostMean"] - r["CI_low"], r["CI_high"] - r["PostMean"]],
            fmt="o", color="#2B6F77", capsize=3, label="Bayesian estimate, 95% interval")
ax.scatter(r["RawRatio"], range(len(r)), marker="x", color="#C0392B", label="Raw actual ÷ expected", zorder=3)
ax.axvline(1, color="grey", lw=1)
ax.set_yticks(range(len(r)))
ax.set_yticklabels(r["RegionName"])
ax.set_xlabel("Regional claim frequency multiplier")
ax.set_title("Regional effects: small regions shrink toward 1.0")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

region_mult = reg["PostMean"].to_dict()
train["freq_final"] = train["freq_glm"] * train["Region"].map(region_mult)
test["freq_final"] = test["freq_glm"] * test["Region"].map(region_mult)

## 10. Benchmark models
To check whether the interpretable GLMs leave accuracy on the table, we compare them with two alternatives:
- **Tweedie GLM:** one model predicts pure premium directly (Tweedie power 1.9 handles the many zeros plus skewed positive costs). Uses region as a regular factor.
- **Gradient boosting:** a Poisson `HistGradientBoostingRegressor` for frequency (raw ages, power, bonus-malus, log density, brand, fuel, region; it finds non-linear effects and interactions by itself), multiplied by the severity GLM.

Pure premium for each frequency–severity model = predicted frequency × predicted severity.

In [ ]:
def region_dummies(data):
    vals = data["Region"].to_numpy()
    base_r = train.groupby("Region")["Exposure"].sum().idxmax()
    return pd.DataFrame({f"Region={r}": (vals == r).astype(np.float64) for r in REGIONS if r != base_r},
                        index=data.index)


Xt_train = pd.concat([X_train, region_dummies(train)], axis=1)
Xt_test = pd.concat([X_test, region_dummies(test)], axis=1)

t0 = time.time()
tweedie = TweedieRegressor(power=1.9, link="log", alpha=1e-4, solver="newton-cholesky", max_iter=300)
tweedie.fit(Xt_train, train["PurePremium"], sample_weight=train["Exposure"])
test["pp_tweedie"] = tweedie.predict(Xt_test)
print(f"Tweedie GLM fitted in {time.time() - t0:.1f}s")

GBM_NUM = ["DrivAge", "VehAge", "VehPower", "BonusMalus", "LogDensity"]
GBM_CAT = ["VehBrand", "VehGas", "Region"]


def gbm_features(data):
    out = data[["DrivAge", "VehAge", "VehPower", "BonusMalus"]].copy()
    out["LogDensity"] = np.log(data["Density"])
    for c in GBM_CAT:
        cats = LEVELS[c] if c in LEVELS else REGIONS
        out[c] = pd.Categorical(data[c], categories=cats).codes
    return out


t0 = time.time()
gbm = HistGradientBoostingRegressor(
    loss="poisson", learning_rate=0.05, max_iter=400, max_leaf_nodes=31, min_samples_leaf=500,
    l2_regularization=1.0, categorical_features=[False] * len(GBM_NUM) + [True] * len(GBM_CAT),
    early_stopping=True, validation_fraction=0.1, random_state=SEED)
gbm.fit(gbm_features(train), train["Frequency"], sample_weight=train["Exposure"])
test["freq_gbm"] = gbm.predict(gbm_features(test))
print(f"Gradient boosting fitted in {time.time() - t0:.1f}s ({gbm.n_iter_} trees)")

test["pp_glm"] = test["freq_glm"] * test["sev_glm"]
test["pp_final"] = test["freq_final"] * test["sev_glm"]
test["pp_gbm"] = test["freq_gbm"] * test["sev_glm"]

## 11. Validation: Gini, Lorenz curve, lift chart
- **Lorenz curve:** sort test policies from lowest to highest predicted pure premium and plot cumulative exposure (x) against cumulative actual claim cost (y). A model that ranks risk well bends the curve far below the diagonal, because the low-predicted policies really do produce few losses.
- **Gini = 1 − 2 × area under the Lorenz curve.** 0 means random ranking. Higher is better at separating cheap from expensive risks.
- **Predicted ÷ actual total:** checks that the model gets the overall price level right (1.00 is perfect).
- **Lift chart:** split test policies into 10 equal-exposure groups by predicted pure premium and compare predicted vs actual cost in each group.

In [ ]:
def lorenz_curve(actual_cost, pred_pp, exposure):
    order = np.argsort(pred_pp, kind="mergesort")
    x = np.r_[0, np.cumsum(exposure[order])]
    y = np.r_[0, np.cumsum(actual_cost[order])]
    return x / x[-1], y / y[-1]


def gini(actual_cost, pred_pp, exposure):
    x, y = lorenz_curve(actual_cost, pred_pp, exposure)
    return 1 - 2 * auc(x, y)


MODELS = {
    "Freq GLM × Sev GLM + Bayesian region (final)": ("pp_final", "freq_final"),
    "Freq GLM × Sev GLM (no region)": ("pp_glm", "freq_glm"),
    "Gradient boosting freq × Sev GLM": ("pp_gbm", "freq_gbm"),
    "Tweedie GLM (single model)": ("pp_tweedie", None),
}
cost, expo = test["ClaimAmount"].to_numpy(), test["Exposure"].to_numpy()
rows = []
for name, (pp_col, f_col) in MODELS.items():
    pp = test[pp_col].to_numpy()
    rows.append({
        "Model": name,
        "Gini": gini(cost, pp, expo),
        "Freq Poisson deviance": (mean_poisson_deviance(test["Frequency"], test[f_col], sample_weight=expo)
                                  if f_col else np.nan),
        "Predicted ÷ actual cost": (pp * expo).sum() / cost.sum(),
    })
comparison = pd.DataFrame(rows)
oracle_gini = gini(cost, test["PurePremium"].to_numpy(), expo)
print(comparison.round(4).to_string(index=False))
print(f"\nFor reference, a perfect 'oracle' ranking would score Gini = {oracle_gini:.3f} "
      "(not reachable: most of claim cost is luck).")

# Lorenz curves, down-sampled to 201 points so the app loads fast.
grid = np.linspace(0, 1, 201)
lorenz_df = pd.DataFrame({"CumExposure": grid, "Random": grid})
plt.figure(figsize=(6.5, 6))
plt.plot(grid, grid, "k--", lw=1, label="Random (Gini 0)")
for name, (pp_col, _) in MODELS.items():
    x, y = lorenz_curve(cost, test[pp_col].to_numpy(), expo)
    lorenz_df[name] = np.interp(grid, x, y)
    plt.plot(grid, lorenz_df[name], label=f"{name.split(' (')[0]}: {gini(cost, test[pp_col].to_numpy(), expo):.3f}")
plt.xlabel("Cumulative share of exposure (lowest predicted risk first)")
plt.ylabel("Cumulative share of claim cost")
plt.title("Lorenz curves on the test set")
plt.legend(fontsize=8)
plt.show()

# Decile lift chart for the final model.
t = test.sort_values("pp_final", kind="mergesort")
t["Decile"] = np.minimum((t["Exposure"].cumsum() / t["Exposure"].sum() * 10).apply(np.ceil), 10).astype(int)
lift = t.groupby("Decile").apply(lambda g: pd.Series({
    "Exposure": g["Exposure"].sum(),
    "PredictedPurePremium": np.average(g["pp_final"], weights=g["Exposure"]),
    "ActualPurePremium": g["ClaimAmount"].sum() / g["Exposure"].sum(),
})).reset_index()
lift["ActualRelativity"] = lift["ActualPurePremium"] / (cost.sum() / expo.sum())
print(lift.round(2).to_string(index=False))
print(f"Top-decile vs bottom-decile actual pure premium: "
      f"{lift['ActualPurePremium'].iloc[-1] / lift['ActualPurePremium'].iloc[0]:.1f}×")

plt.figure(figsize=(8, 4))
w = 0.4
plt.bar(lift["Decile"] - w / 2, lift["PredictedPurePremium"], w, label="Predicted", color="#2B6F77")
plt.bar(lift["Decile"] + w / 2, lift["ActualPurePremium"], w, label="Actual", color="#E0A458")
plt.xticks(range(1, 11))
plt.xlabel("Decile of predicted pure premium (1 = lowest risk)")
plt.ylabel("€ per policy-year")
plt.title("Lift chart: final model, test set")
plt.legend()
plt.show()

## 12. Relativity table and top risk drivers
A **relativity** is how much a level changes the premium compared with the baseline level (1.00). Because frequency and severity multiply, the **pure premium relativity = frequency relativity × severity relativity**.

Risk drivers are ranked by their **spread**: highest relativity ÷ lowest relativity across levels with at least 0.5% of exposure, so tiny levels don't exaggerate a factor.

In [ ]:
FACTOR_LABELS = {
    "DrivAgeBand": "Driver age", "VehAgeBand": "Vehicle age", "VehPowerBand": "Vehicle power",
    "BonusMalusBand": "Bonus-malus", "DensityBand": "Population density",
    "VehBrand": "Vehicle brand", "VehGas": "Fuel type", "Region": "Region",
}
rel_rows = []
for f in FACTORS:
    expo_share = train.groupby(f)["Exposure"].sum() / train["Exposure"].sum()
    for lv in LEVELS[f]:
        bf = 0.0 if lv == BASE[f] else FREQ_COEF[f"{f}={lv}"]
        bs = 0.0 if lv == BASE[f] else SEV_COEF[f"{f}={lv}"]
        rel_rows.append({"Factor": FACTOR_LABELS[f], "Level": lv, "Baseline": lv == BASE[f],
                         "ExposureShare": float(expo_share.get(lv, 0.0)),
                         "FrequencyRelativity": np.exp(bf), "SeverityRelativity": np.exp(bs),
                         "PurePremiumRelativity": np.exp(bf + bs)})
for rcode, row in reg.iterrows():
    rel_rows.append({"Factor": "Region", "Level": f"{rcode} {row['RegionName']}", "Baseline": False,
                     "ExposureShare": row["ExposureShare"], "FrequencyRelativity": row["PostMean"],
                     "SeverityRelativity": 1.0, "PurePremiumRelativity": row["PostMean"]})
relativity = pd.DataFrame(rel_rows)

drivers = (relativity.loc[relativity["ExposureShare"] >= 0.005]
           .groupby("Factor")["PurePremiumRelativity"].agg(["min", "max"]))
drivers["Spread"] = drivers["max"] / drivers["min"]
drivers = drivers.sort_values("Spread", ascending=False)
print("Risk drivers ranked by spread of pure premium relativity:")
print(drivers.round(3).to_string())

TOP_DRIVERS = []
print("\nTop three risk drivers:")
for fac in drivers.index[:3]:
    sub = relativity.loc[(relativity["Factor"] == fac) & (relativity["ExposureShare"] >= 0.005)]
    hi = sub.loc[sub["PurePremiumRelativity"].idxmax()]
    lo = sub.loc[sub["PurePremiumRelativity"].idxmin()]
    line = (f"{fac}: {hi['Level']} = {hi['PurePremiumRelativity']:.2f}× vs {lo['Level']} = "
            f"{lo['PurePremiumRelativity']:.2f}× (spread {drivers.loc[fac, 'Spread']:.1f}×)")
    TOP_DRIVERS.append(line)
    print(" •", line)

print("\n" + relativity.loc[relativity["Factor"] != "Region"]
      .round(3).to_string(index=False))

## 13. Save the model files for the Streamlit app
The app never trains anything or runs PyMC. It only reads these small files:
- `model_artifacts.json`: GLM coefficients, intercepts, binning rules, baseline levels, dropdown options, portfolio averages
- `regional_estimates.csv`: Bayesian regional multipliers with 95% credible intervals
- `relativity_table.csv`, `model_comparison.csv`, `lorenz_curves.csv`, `lift_chart.csv`: validation results for the Model Insights tab

Then the cell zips the folder and downloads it. In GitHub, put the unzipped `artifacts/` folder next to `app.py`.

In [ ]:
def to_py(o):
    """Convert numpy types so json.dump works."""
    if isinstance(o, dict):
        return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o


def coef_table(coefs):
    return {f: {lv: (0.0 if lv == BASE[f] else float(coefs[f"{f}={lv}"])) for lv in LEVELS[f]} for f in FACTORS}


train_avg_pp = float(train["ClaimAmount"].sum() / train["Exposure"].sum())
artifacts = {
    "description": "Auto insurance pure premium model trained on freMTPL2 (French MTPL) data",
    "created": time.strftime("%Y-%m-%d %H:%M"),
    "seed": SEED,
    "band_rules": {k: {"source": v[0], "rules": [list(r) for r in v[1]]} for k, v in BAND_RULES.items()},
    "factors": FACTORS,
    "factor_labels": FACTOR_LABELS,
    "levels": LEVELS,
    "base_levels": BASE,
    "frequency_glm": {"family": "Poisson", "link": "log", "intercept": FREQ_INTERCEPT,
                      "coefficients": coef_table(FREQ_COEF)},
    "severity_glm": {"family": "Gamma", "link": "log", "intercept": SEV_INTERCEPT,
                     "coefficients": coef_table(SEV_COEF)},
    "region_method": REGION_METHOD,
    "regions": REGIONS,
    "region_names": {r: REGION_NAMES.get(r, r) for r in REGIONS},
    "portfolio": {
        "claim_frequency": float(train["ClaimNb"].sum() / train["Exposure"].sum()),
        "avg_severity": float(train["ClaimAmount"].sum() / train["ClaimNb"].sum()),
        "pure_premium": train_avg_pp,
        "n_policies_total": portfolio["n_policies"],
        "n_policies_train": int(len(train)),
    },
    "input_ranges": {c: [float(df[c].min()), float(df[c].max())]
                     for c in ["DrivAge", "VehAge", "VehPower", "BonusMalus", "Density"]},
    "cleaning_log": log,
}
with open(f"{OUT}/model_artifacts.json", "w", encoding="utf-8") as fh:
    json.dump(to_py(artifacts), fh, indent=2, ensure_ascii=False)

reg_out = reg.reset_index().rename(columns={"index": "Region"})
reg_out[["Region", "RegionName", "Exposure", "ExposureShare", "Claims", "Expected", "RawRatio",
         "PostMean", "CI_low", "CI_high"]].to_csv(f"{OUT}/regional_estimates.csv", index=False)
relativity.to_csv(f"{OUT}/relativity_table.csv", index=False)
comparison.to_csv(f"{OUT}/model_comparison.csv", index=False)
lorenz_df.to_csv(f"{OUT}/lorenz_curves.csv", index=False)
lift.to_csv(f"{OUT}/lift_chart.csv", index=False)

# Sanity check: recompute one test policy's premium from the saved JSON, the way the app does.
with open(f"{OUT}/model_artifacts.json", encoding="utf-8") as fh:
    A = json.load(fh)
p = test.iloc[0]
lin_f = A["frequency_glm"]["intercept"] + sum(A["frequency_glm"]["coefficients"][f][p[f]] for f in FACTORS)
lin_s = A["severity_glm"]["intercept"] + sum(A["severity_glm"]["coefficients"][f][p[f]] for f in FACTORS)
pp_from_json = np.exp(lin_f) * region_mult[p["Region"]] * np.exp(lin_s)
print(f"Saved-file check: notebook €{p['pp_final']:.4f} vs JSON €{pp_from_json:.4f}")
assert abs(pp_from_json - p["pp_final"]) < 1e-6 * max(1, p["pp_final"])

for fname in sorted(os.listdir(OUT)):
    print(f"  {fname:28s} {os.path.getsize(os.path.join(OUT, fname)) / 1024:7.1f} KB")

import shutil
shutil.make_archive("artifacts", "zip", OUT)
try:
    from google.colab import files
    files.download("artifacts.zip")
except ImportError:
    print("Not running in Colab: artifacts.zip is in the working folder.")

## 14. Results summary
Copy this output into your README and send it to Claude for your resume. These are your real numbers.

In [ ]:
best = comparison.sort_values("Gini", ascending=False).iloc[0]
final = comparison.iloc[0]


def table_row(r):
    dev = r["Freq Poisson deviance"]
    dev_txt = "" if pd.isna(dev) else f"{dev:.4f}"
    return f"| {r['Model']} | {r['Gini']:.3f} | {dev_txt} | {r['Predicted ÷ actual cost']:.3f} |"


summary = f"""## Results (test set, {len(test):,} policies)

**Data:** {portfolio['n_policies']:,} policies after cleaning, {portfolio['total_exposure_years']:,.0f} policy-years,
{portfolio['n_claims']:,} claims. Claim frequency {portfolio['claim_frequency']:.4f} per year,
average severity €{portfolio['avg_severity']:,.0f}, pure premium €{portfolio['pure_premium']:,.2f} per policy-year.

| Model | Gini | Freq Poisson deviance | Predicted ÷ actual cost |
|---|---|---|---|
""" + "\n".join(table_row(r) for _, r in comparison.iterrows()) + f"""

**Top three risk drivers (pure premium relativity spread):**
""" + "\n".join(f"- {d}" for d in TOP_DRIVERS) + f"""

**Regional Bayesian estimates ({REGION_METHOD}):** multipliers range from
{reg['PostMean'].min():.2f} ({reg.loc[reg['PostMean'].idxmin(), 'RegionName']},
95% CI {reg.loc[reg['PostMean'].idxmin(), 'CI_low']:.2f}–{reg.loc[reg['PostMean'].idxmin(), 'CI_high']:.2f})
to {reg['PostMean'].max():.2f} ({reg.loc[reg['PostMean'].idxmax(), 'RegionName']},
95% CI {reg.loc[reg['PostMean'].idxmax(), 'CI_low']:.2f}–{reg.loc[reg['PostMean'].idxmax(), 'CI_high']:.2f}).

**Lift:** the highest-risk decile has {lift['ActualPurePremium'].iloc[-1] / lift['ActualPurePremium'].iloc[0]:.1f}×
the actual pure premium of the lowest-risk decile.

**Best Gini:** {best['Model']} ({best['Gini']:.3f}). Final interpretable model Gini: {final['Gini']:.3f}.
"""
print(summary)
with open(f"{OUT}/results_summary.md", "w", encoding="utf-8") as fh:
    fh.write(summary)